# Per-Stock Training

Trains one LSTM per stock on the 49 tickers with pre-computed sentiment.
Matches the reference paper's methodology:

- **Window**: 20 trading days, sentiment-gated (anchor day must have news)
- **Architecture**: LSTM(input=32, hidden=32, layers=2) with sentiment projected 768→16
- **Hyperparameters**: lr=1e-3, StepLR(10, 0.1), 150 epochs, batch=16, no early stopping
- **Split**: train < 2023-04, val = last 10% of pre-June-2023, test ≥ 2023-06

Prerequisites:
- `data/prices/data/historical-prices/<SYMBOL>/<YEAR>.csv`
- `data/sentiment/data/sentiment/<SYMBOL>.parquet`

In [ ]:
from __future__ import annotations

import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import src
from src.log import setup_logging

setup_logging()
logger = logging.getLogger("train_per_stock")

: 

## Config

In [ ]:
from src.training import ComputeConfig, TrainingConfig

CUTOFF      = "2023-06-01"
VAL_FRAC    = 0.1
PRICE_YEARS = list(range(2018, 2025))
SEED        = 42

config = TrainingConfig.reference(seed=SEED)  # lr=1e-3, StepLR, 150 epochs, window=20, batch=16
compute_config = ComputeConfig(num_workers=0)
compute_config.setup()

print(f"Device : {compute_config.device}")
print(f"Window : {config.window}")
print(f"LR     : {config.lr}")
print(f"Epochs : {config.n_epochs}")
print(f"Sched  : {config.scheduler} (step={config.step_size}, γ={config.gamma})")

## Discover tickers with sentiment data

In [ ]:
from src.repositories.prices import PriceRepository
from src.repositories.sentiment import SentimentRepository

# Actual data paths (nested inside data/ due to DVC structure)
PRICE_DIR = Path("../data/historical-prices/prices/data/historical-prices")
SENT_DIR  = Path("../data/sentiment/data/sentiment")

price_repo = PriceRepository(data_dir=PRICE_DIR)
sent_repo  = SentimentRepository(data_dir=SENT_DIR)

# Find tickers that have both price and sentiment data
sent_tickers = sorted(t for t in (
    f.stem for f in SENT_DIR.glob("*.parquet")
))
tickers = []
for t in sent_tickers:
    try:
        price_repo.load(t, 2024)
        tickers.append(t)
    except FileNotFoundError:
        print(f"  Skipping {t}: no price data")

print(f"\nTickers with both prices and sentiment: {len(tickers)}")
print(tickers)

## Train one model per stock

For each ticker:
1. Load prices + sentiment
2. Build `StockDataset` (computes tech factors, aligns sentiment)
3. Build sentiment-gated DataLoaders (only windows where anchor day has news)
4. Create fresh LSTM (32 hidden, 768→16 sentiment projection)
5. Train 150 epochs with reference hyperparameters
6. Evaluate on test set (post-cutoff) with bootstrap CIs
7. Save checkpoint + test predictions

In [ ]:
from src.features.dataset import StockDataset, build_per_stock_loaders
from src.model.lstm import SentimentLSTM
from src.model.trainer import Trainer
from src.repositories.models import ModelRepository

model_repo = ModelRepository()

results: list[dict] = []
failed: list[str] = []

for i, ticker in enumerate(tickers):
    print(f"\n{'='*60}")
    print(f"[{i+1}/{len(tickers)}] {ticker}")
    print(f"{'='*60}")

    # --- Load data ---
    try:
        price_df = price_repo.load_years(ticker, PRICE_YEARS)
    except FileNotFoundError:
        print(f"  No price data, skipping")
        failed.append(ticker)
        continue

    sentiment_df = sent_repo.load(ticker)

    # --- Build dataset ---
    try:
        ds = StockDataset(
            symbol=ticker,
            price_df=price_df,
            sentiment_df=sentiment_df,
            window=config.window,
        )
    except RuntimeError as exc:
        print(f"  Dataset error: {exc}")
        failed.append(ticker)
        continue

    # --- Build sentiment-gated loaders ---
    train_loader, val_loader, test_loader = build_per_stock_loaders(
        ds,
        cutoff=CUTOFF,
        val_frac=VAL_FRAC,
        batch_size=config.batch_size,
    )

    n_train = len(train_loader.dataset)
    n_val   = len(val_loader.dataset)
    n_test  = len(test_loader.dataset)

    if n_train == 0:
        print(f"  No training data, skipping")
        failed.append(ticker)
        continue

    print(f"  Windows — train: {n_train}, val: {n_val}, test: {n_test}")

    # --- Create model ---
    model = SentimentLSTM(
        n_factors=16,
        sentiment_dim=768,
        sent_proj_dim=16,
        hidden_size=32,
        num_layers=2,
        dropout=0.2,
        n_sentiment_probs=0,
    )

    # --- Train ---
    trainer = Trainer(model, config, compute_config)
    train_result = trainer.fit(train_loader, val_loader)

    print(
        f"  Best epoch: {train_result.best_epoch} | "
        f"val_loss: {train_result.best_val_loss:.4f} | "
        f"val_auc: {train_result.best_val_auc:.4f}"
    )

    # --- Evaluate test set ---
    if n_test > 0:
        eval_result = trainer.bootstrap_evaluate(test_loader, n_bootstrap=1000, seed=SEED)
        print(
            f"  Test AUC:  {eval_result.auc_mean:.3f} "
            f"[{eval_result.auc_ci_low:.3f}, {eval_result.auc_ci_high:.3f}]"
        )
        print(
            f"  Test Acc:  {eval_result.accuracy_mean:.3f} "
            f"[{eval_result.accuracy_ci_low:.3f}, {eval_result.accuracy_ci_high:.3f}]"
        )
    else:
        eval_result = None
        print("  No test data")

    # --- Save checkpoint ---
    ckpt_name = f"per_stock_lstm_{ticker}"
    model_repo.save(
        ckpt_name,
        model,
        {
            "ticker": ticker,
            "window": config.window,
            "n_train": n_train,
            "n_val": n_val,
            "n_test": n_test,
            "best_epoch": train_result.best_epoch,
            "best_val_loss": train_result.best_val_loss,
            "best_val_auc": train_result.best_val_auc,
            "test_auc": eval_result.auc_mean if eval_result else None,
            "test_accuracy": eval_result.accuracy_mean if eval_result else None,
            "history": train_result.history,
        },
    )

    results.append({
        "ticker": ticker,
        "n_train": n_train,
        "n_test": n_test,
        "best_epoch": train_result.best_epoch,
        "val_loss": train_result.best_val_loss,
        "val_auc": train_result.best_val_auc,
        "test_auc": eval_result.auc_mean if eval_result else None,
        "test_acc": eval_result.accuracy_mean if eval_result else None,
    })

print(f"\n\nDone. Trained: {len(results)}, Failed: {len(failed)}")
if failed:
    print(f"Failed tickers: {failed}")

## Results summary

In [ ]:
df_results = pd.DataFrame(results).sort_values("test_auc", ascending=False)
print(df_results.to_string(index=False, float_format="%.3f"))
print(f"\nMean test AUC: {df_results['test_auc'].mean():.3f}")
print(f"Mean test Acc: {df_results['test_acc'].mean():.3f}")
print(f"Stocks with AUC > 0.5: {(df_results['test_auc'] > 0.5).sum()} / {len(df_results)}")